In [6]:
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent

import pandas as pd

# ============================================================
# 1. 데이터 불러오기
# ============================================================

train = pd.read_parquet(PROJECT_ROOT / 'parqueted_data' / 'train_split.parquet')
val = pd.read_parquet(PROJECT_ROOT / 'parqueted_data' / 'val_split.parquet')
test = pd.read_parquet(PROJECT_ROOT / 'parqueted_data' / 'test.parquet')

from sklearn import set_config
set_config(display='text')
# scikit-learn의 HTML 출력 과정에서 발생한 cp949 간 UTF-8 충돌 때문에 넣는 코드

In [7]:
# ============================================================
# 2. 컬럼군 정의
# ============================================================

target_col = 'target_maxrise_6h'  # 모델링 학습부터는 반드시 빼내야 함

# 수위 컬럼군
wl_cols = [
    'wl_t',
    'wl_t_minus_1h',
    'wl_t_minus_3h',
    'wl_t_minus_6h',
    'wl_t_minus_12h',
    'wl_t_minus_24h',
]

# 강수 컬럼군
rain_cols = [
    # IMERG강우
    'imerg_rain_1h',
    'imerg_rain_6h',
    'imerg_rain_24h',
    # 지상강우
    'sfc_rain_1h',
    'sfc_rain_6h',
    'sfc_rain_24h',
    # 레이더 반사도
    'radar_reflectivity_mean',
    'radar_reflectivity_p95',
    'radar_reflectivity_ge20_fraction'
]

# 환경 컬럼군
env_cols = [
    'cat_area',       # km²
    'wsarea',         # km²
    'mean_elevation', # m
    'mean_slope',     # °
    'urban_ratio',    # %
    'forest_ratio',   # %
    'stream_order'
]

num_cols = wl_cols + rain_cols + env_cols
# print(len(wl_cols), len(rain_cols), len(env_cols), len(num_cols))

In [8]:
# ============================================================
# 3. 종속변수 설정
# ============================================================

y_train = train[target_col]
y_val = val[target_col]

In [9]:
# =====================================================================
# 4. cat_area, wsarea에서 이상값으로 의심된 데이터 포함여부에 따른 모델링
# =====================================================================
# 면적 이상값은 오측정된 값은 아니고, 대유역을 포함한 관측소 1개소 였음
# 무작정 제거하기보다는 포함여부에 따른 모델을 비교하고자 함

# 4-1. 해당 관측소가 train, val, test 셋 중 어디에 속해있는지 확인

# train셋 이상치 확인
print("train_dataset")
display(pd.concat(
    [
        train['cat_area']
            .drop_duplicates()
            .sort_values(ascending = False)
            .reset_index(drop=True),

        train['wsarea']
            .drop_duplicates()
            .sort_values(ascending = False)
            .reset_index(drop=True)
    ],
    axis = 1
))

# val셋
print("val_dataset")
display(pd.concat(
    [
        val['cat_area']
            .drop_duplicates()
            .sort_values(ascending = False)
            .reset_index(drop=True),

        val['wsarea']
            .drop_duplicates()
            .sort_values(ascending = False)
            .reset_index(drop=True)
    ], 
    axis = 1
))

# test셋
print("test_dataset")
display(pd.concat(
    [
        test['cat_area']
            .drop_duplicates()
            .sort_values(ascending=False)
            .reset_index(drop=True),

        test['wsarea']
            .drop_duplicates()
            .sort_values(ascending=False)
            .reset_index(drop=True)
    ],
    axis = 1
))

# 이상치는 train, val, test 모두에 포함됨

train_dataset


,cat_area,wsarea
0,50.6527,29687.6916
1,18.6391,11684.3950
2,15.8718,10766.0188
3,11.8928,9037.7681
4,11.3891,7888.0657
...,...,...
324,2.4657,9.8050
325,2.3503,7.3666
326,2.3451,6.3167
327,2.3403,5.7516


val_dataset


,cat_area,wsarea
0,50.6527,29687.6916
1,18.6391,11684.3950
2,15.8718,10766.0188
3,11.8928,9037.7681
4,11.3891,7888.0657
...,...,...
324,2.4657,9.8050
325,2.3503,7.3666
326,2.3451,6.3167
327,2.3403,5.7516


test_dataset


,cat_area,wsarea
0,50.6527,29687.6916
1,18.6391,11684.3950
2,15.8718,10766.0188
3,11.8928,9037.7681
4,11.3891,7888.0657
...,...,...
324,2.4657,9.8050
325,2.3503,7.3666
326,2.3451,6.3167
327,2.3403,5.7516


In [10]:
# cat_area, wsarea 이상치 기준
# cat_area는 50 이상
# wsarea는 20000 이상

# 4-2. train셋만 극단치 제거 후 OHE
from sklearn.preprocessing import OneHotEncoder

cond_cat_out = train['cat_area'] > 50
cond_wsarea_out = train['wsarea'] > 20000
cond_select = ~(cond_cat_out | cond_wsarea_out)
# extreme_station = train.loc[cond_cat_out | cond_wsarea_out]['station_id']
# normal_station = train.loc[~(cond_cat_out | cond_wsarea_out)]['station_id']

# print(extreme_station.unique())  # 'ST0051'

ohe = OneHotEncoder(
    handle_unknown='ignore',
    sparse_output=True
)

ohe.fit(train.loc[cond_select, ['station_id']])

train_normal_ohe = ohe.transform(train.loc[cond_select, ['station_id']])
val_ohe = ohe.transform(val[['station_id']])


# 합치기
from scipy.sparse import hstack
X_train_normal_ohe = hstack([
    train.loc[cond_select, num_cols],
    train_normal_ohe
])

X_val_ohe = hstack([
    val[num_cols],
    val_ohe
])

In [11]:
# 4-3. X_train_normal_ohe에 맞는 y_train_normal 만들기

# X_train_normal_ohe는 행 개수가 변했기때문에 종속변수도 재설정 필요
y_train_normal = train.loc[cond_select, target_col]

In [12]:
# 4-4 극단값 빼고 모델링
from lightgbm import LGBMRegressor
from sklearn.metrics import root_mean_squared_error

model_no_ext = LGBMRegressor(
    random_state=42
)

model_no_ext.fit(
    X_train_normal_ohe,
    y_train_normal
)

pred_no_ext = model_no_ext.predict(X_val_ohe)

rmse_no_ext = root_mean_squared_error(y_val, pred_no_ext)

print(f"cat_area, wsarea 극단값 제외 모델 rmse: {rmse_no_ext}")


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.045215 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 6460
[LightGBM] [Info] Number of data points in the train set: 614952, number of used features: 486
[LightGBM] [Info] Start training from score 0.111444


c:\Users\20162180\AppData\Local\Programs\Python\Python314\Lib\site-packages\lightgbm\basic.py:1166: UserWarning: Converting data to scipy sparse matrix.
  _log_warning("Converting data to scipy sparse matrix.")


cat_area, wsarea 극단값 제외 모델 rmse: 0.2126643972778919
